In [2]:
import boto3
import numpy as np
import pandas as pd
import json
import io
import matplotlib.pyplot as plt

In [3]:
bucket = "signal-platform-dev-471112934830"
key = "raw/domain=ligo/"
meta_key = key.replace(".npy", ".json")

In [4]:
s3 = boto3.client("s3", region_name="us-east-1")

/Users/alilordifar/opt/anaconda3/envs/gw/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [6]:
bronze_df = pd.read_parquet(
    "s3://signal-platform-dev-471112934830/bronze/domain=ligo/",
    partitioning=None
)


In [7]:
bronze_df.shape[0]

14336

In [8]:
bronze_df.head()

,window_id,source_id,asset_key,asset_start_time_utc,start_idx,end_idx,window_duration,window_num_samples,window_start_time_utc,sample_rate_hz,domain_metadata,ingestion_ts,event_date
0,0,H1,raw/domain=ligo/source_id=H1/year=2015/month=0...,1.442224e+09,0,8192,2.0,8192,1.442224e+09,4096.0,"[(gps_start, 1126259462), (gps_end, 1126263558)]",2026-08-11 00:55:01.809,2015-09-14
1,1,H1,raw/domain=ligo/source_id=H1/year=2015/month=0...,1.442224e+09,8192,16384,2.0,8192,1.442224e+09,4096.0,"[(gps_start, 1126259462), (gps_end, 1126263558)]",2026-08-11 00:55:01.809,2015-09-14
2,2,H1,raw/domain=ligo/source_id=H1/year=2015/month=0...,1.442224e+09,16384,24576,2.0,8192,1.442224e+09,4096.0,"[(gps_start, 1126259462), (gps_end, 1126263558)]",2026-08-11 00:55:01.809,2015-09-14
3,3,H1,raw/domain=ligo/source_id=H1/year=2015/month=0...,1.442224e+09,24576,32768,2.0,8192,1.442224e+09,4096.0,"[(gps_start, 1126259462), (gps_end, 1126263558)]",2026-08-11 00:55:01.809,2015-09-14
4,4,H1,raw/domain=ligo/source_id=H1/year=2015/month=0...,1.442224e+09,32768,40960,2.0,8192,1.442224e+09,4096.0,"[(gps_start, 1126259462), (gps_end, 1126263558)]",2026-08-11 00:55:01.809,2015-09-14


In [22]:
bronze_df["event_date"].nunique()

7

In [10]:
strains = {}
for asset_key in bronze_df["asset_key"].unique():
    obj = s3.get_object(Bucket=bucket, Key=asset_key)
    strains[asset_key] = np.load(io.BytesIO(obj["Body"].read()))

In [11]:
strains

{'raw/domain=ligo/source_id=H1/year=2015/month=09/day=14/H1_1442224245.000_4096.000.npy': array([5.16251157e-20, 3.72676369e-20, 2.76847613e-20, ...,
        2.65035515e-19, 2.39260773e-19, 2.42696492e-19]),
 'raw/domain=ligo/source_id=H1/year=2015/month=10/day=12/H1_1444643183.000_4096.000.npy': array([-1.56866647e-19, -1.55880878e-19, -1.49021781e-19, ...,
        -1.15658460e-19, -1.06841049e-19, -9.93369368e-20]),
 'raw/domain=ligo/source_id=H1/year=2015/month=12/day=26/H1_1451099085.000_4096.000.npy': array([            nan,             nan,             nan, ...,
        -7.91898881e-20, -9.78563093e-20, -1.02612436e-19]),
 'raw/domain=ligo/source_id=H1/year=2017/month=01/day=04/H1_1483522670.000_4096.000.npy': array([-6.29341279e-19, -6.45170044e-19, -6.36808585e-19, ...,
         3.52198640e-19,  3.70889469e-19,  3.94879628e-19]),
 'raw/domain=ligo/source_id=H1/year=2017/month=06/day=08/H1_1496885228.000_4096.000.npy': array([-5.87727681e-19, -6.95406506e-19, -7.11660593e-19, ..

In [40]:
flags = []
for asset_key, group in bronze_df.groupby("asset_key"):
    raw = strains[asset_key]
    global_std = np.nanstd(raw)

    for _, row in group.iterrows():
        window_raw = raw[row["start_idx"]:row["end_idx"]]
        has_nan = np.isnan(window_raw).any()
        is_flat = np.nanstd(window_raw) < (global_std * 1e-3)
        mean = np.nanmean(window_raw)
        std = np.nanstd(window_raw)
        has_outlier = np.any(np.abs(window_raw - mean) > 3 * std)

        flags.append({
            "window_id": row["window_id"],
            "is_nan": bool(has_nan),
            "is_flat": bool(is_flat),
            "has_outlier": bool(has_outlier)
        })

/Users/alilordifar/opt/anaconda3/envs/gw/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1878: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/var/folders/2j/8rkmt49n20zd441vp_sg9lym0000gn/T/ipykernel_15318/355411383.py:10: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(window_raw)
/Users/alilordifar/opt/anaconda3/envs/gw/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1878: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/var/folders/2j/8rkmt49n20zd441vp_sg9lym0000gn/T/ipykernel_15318/355411383.py:10: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(window_raw)


In [38]:
# Look for the first window that has a NaN
for flag in flags:
    if flag["is_nan"] == "true":
        print(f"Found NaN in window: {flag['window_id']}")
        break  # Stops after finding the first one
else:
    # This runs only if the loop finishes without hitting 'break'
    print("No NaNs found anywhere.")


No NaNs found anywhere.


In [41]:
#flags